# Calligraphy Dataset Preprocessing to 128x128 PNG

This notebook only preprocesses local calligraphy single-character images. It reads the six local bundles, writes white-background black-ink grayscale PNG files at 128x128, and records metadata/failures.

This version uses **balanced v2 background cleanup**: clean enough for gray paper backgrounds, but conservative enough to preserve faint ink, flying white, thin strokes, and brush texture.

No evaluation, model training, train/val/test split, accuracy, F1, or confusion matrix is included.


## 1. Configuration

All input and output paths stay inside `/Users/lainantian/Developer/Caligraphy`. This v2 test run uses 100 target rows and intentionally includes 草書、章草、行書、篆書、楷書 plus suspected black-background and gray-background examples.


In [ ]:
from pathlib import Path
from collections import defaultdict, deque
import math
import os
import re
import shutil
import traceback
import unicodedata

import numpy as np
import pandas as pd
from PIL import Image, ImageOps, ImageDraw, UnidentifiedImageError
from tqdm.auto import tqdm
import matplotlib.pyplot as plt

try:
    import cv2
    HAS_CV2 = True
except Exception:
    HAS_CV2 = False

try:
    from scipy import ndimage as ndi
    HAS_SCIPY = True
except Exception:
    HAS_SCIPY = False

ROOT = Path('/Users/lainantian/Developer/Caligraphy')
OUTPUT_ROOT = ROOT / 'Caligraphy_preprocessed_128_test100_v2'
OUTPUT_IMAGE_ROOT = OUTPUT_ROOT / 'images'

RUN_FULL = False
SAMPLE_SIZE = 100
TARGET_SIZE = 128
TINY_SPECK_AREA = 8
OUTSIDE_BBOX_SPECK_AREA = 25
BBOX_MIN_COMPONENT_AREA = 8
CLEAN_OUTPUT_ROOT = True

BUNDLE_NAMES = [
    'common_c_bundle',
    'common_g_bundle 2',
    'common_h_bundle',
    'common_i_bundle',
    'common_j_bundle',
    'less_common_b_bundle',
]

ORIGINAL_COLUMNS = [
    'query_char', 'style_code', 'style_name', 'era', 'author',
    'work_title', 'detail_title', 'image_url', 'thumb_url'
]

STYLE_MAP = {
    '楷书': '楷書',
    '篆书': '篆書',
    '草书': '草書',
    '章草': '草書',
    '行书': '行書',
    '隶书': '隸書',
}
STYLE_LABEL_ORDER = ['楷書', '篆書', '草書', '行書', '隸書']
IMAGE_EXTENSIONS = {'.jpg', '.jpeg', '.png', '.webp'}
VECTOR_HINT_PHRASES = ['如果不满意这么大', '矢量部分', '找到原大的']
TEXT_COLUMNS = ORIGINAL_COLUMNS

if CLEAN_OUTPUT_ROOT and OUTPUT_ROOT.exists():
    shutil.rmtree(OUTPUT_ROOT)

OUTPUT_IMAGE_ROOT.mkdir(parents=True, exist_ok=True)
for style_label in STYLE_LABEL_ORDER:
    (OUTPUT_IMAGE_ROOT / style_label).mkdir(parents=True, exist_ok=True)

print(f'ROOT: {ROOT}')
print(f'OUTPUT_ROOT: {OUTPUT_ROOT}')
print(f'TARGET_SIZE: {TARGET_SIZE}')
print(f'RUN_FULL: {RUN_FULL}')
print(f'SAMPLE_SIZE: {SAMPLE_SIZE}')
print(f'OpenCV available: {HAS_CV2}; SciPy available: {HAS_SCIPY}')


## 2. Metadata and filename helpers

These helpers normalize metadata text, skip vector-hint images without OCR, and create safe output filenames.


In [ ]:
def norm_text(value) -> str:
    if value is None:
        return ''
    try:
        if pd.isna(value):
            return ''
    except TypeError:
        pass
    text = unicodedata.normalize('NFKC', str(value)).strip()
    if text.lower() in {'nan', 'none', 'nat'}:
        return ''
    return text


def work_name_for_filename(value) -> str:
    return norm_text(value) or '未标注作品'


def contains_vector_hint(*values) -> bool:
    combined = ' '.join(norm_text(v) for v in values)
    return any(phrase in combined for phrase in VECTOR_HINT_PHRASES)


def safe_filename_part(value, default='unknown', max_len=48) -> str:
    text = norm_text(value) or default
    invalid_chars = set('/\\:*?"<>|')
    text = ''.join('_' if ch in invalid_chars or ord(ch) < 32 else ch for ch in text)
    text = re.sub(r'\s+', '_', text)
    text = re.sub(r'_+', '_', text).strip('._ ')
    return (text or default)[:max_len]


def parse_download_filename(path: Path) -> dict:
    parts = path.stem.split('_')
    serial_text = parts[0] if len(parts) > 0 else ''
    try:
        serial_num = int(serial_text)
    except ValueError:
        serial_num = 10**12

    return {
        'path': path,
        'serial_num': serial_num,
        'file_query_char': parts[1] if len(parts) > 1 else '',
        'file_style_name': parts[2] if len(parts) > 2 else path.parent.name,
        'file_author': parts[3] if len(parts) > 3 else '',
        'file_work_title': '_'.join(parts[4:]) if len(parts) > 4 else '',
        'filename': path.name,
    }


def row_text_values(row) -> list:
    return [row.get(col, '') for col in TEXT_COLUMNS]


def output_filename(row) -> str:
    global_index = int(row['global_index'])
    query_char = safe_filename_part(row.get('query_char'), default='unknown_char', max_len=12)
    style_label = safe_filename_part(row.get('style_label'), default='unknown_style', max_len=12)
    author = safe_filename_part(row.get('author'), default='unknown_author', max_len=48)
    return f'{global_index:08d}_{query_char}_{style_label}_{author}.png'


## 3. Load CSV rows and match local images

For each bundle, image files are scanned from `{bundle}/*_downloads/{style_name}/`. Matching uses filename metadata first, then falls back to looser keys and style-folder order.


In [ ]:
def make_file_indexes(file_records):
    exact = defaultdict(deque)
    char_author = defaultdict(deque)
    char_style = defaultdict(deque)
    by_style = defaultdict(deque)

    for rec in file_records:
        query_char = norm_text(rec['file_query_char'])
        style_name = norm_text(rec['file_style_name'])
        author = norm_text(rec['file_author'])
        work_title = work_name_for_filename(rec['file_work_title'])

        exact[(query_char, style_name, author, work_title)].append(rec)
        char_author[(query_char, style_name, author)].append(rec)
        char_style[(query_char, style_name)].append(rec)
        by_style[style_name].append(rec)

    return exact, char_author, char_style, by_style


def pop_unused(queue, used_paths):
    while queue:
        rec = queue.popleft()
        if rec['path'] not in used_paths:
            used_paths.add(rec['path'])
            return rec
    return None


def match_image_for_row(row, indexes, used_paths):
    exact, char_author, char_style, by_style = indexes

    query_char = norm_text(row.get('query_char'))
    style_name = norm_text(row.get('style_name'))
    author = norm_text(row.get('author'))
    work_title = work_name_for_filename(row.get('work_title'))

    attempts = [
        ('filename_query_style_author_work', exact[(query_char, style_name, author, work_title)]),
        ('fallback_query_style_author', char_author[(query_char, style_name, author)]),
        ('fallback_query_char_style', char_style[(query_char, style_name)]),
        ('fallback_style_folder_order', by_style[style_name]),
    ]

    for matched_by, queue in attempts:
        rec = pop_unused(queue, used_paths)
        if rec is not None:
            return rec, matched_by

    return None, 'no_image_found'


def read_bundle_csv(bundle_name: str, global_start: int):
    bundle_path = ROOT / bundle_name
    csv_files = sorted(bundle_path.glob('*.csv'))
    if not csv_files:
        raise FileNotFoundError(f'No CSV found in {bundle_path}')

    csv_path = csv_files[0]
    df = pd.read_csv(csv_path, encoding='utf-8-sig')
    df.columns = [str(col).strip().lstrip('\ufeff') for col in df.columns]

    missing_cols = [col for col in ORIGINAL_COLUMNS if col not in df.columns]
    if missing_cols:
        raise ValueError(f'{csv_path} missing columns: {missing_cols}')

    df = df[ORIGINAL_COLUMNS].copy()
    df.insert(0, 'global_index', range(global_start, global_start + len(df)))
    df['bundle'] = bundle_name
    df['style_label'] = df['style_name'].map(STYLE_MAP)
    return df


def scan_bundle_image_files(bundle_name: str):
    bundle_path = ROOT / bundle_name
    download_dirs = sorted(bundle_path.glob('*_downloads'))
    if not download_dirs:
        raise FileNotFoundError(f'No *_downloads directory found in {bundle_path}')

    download_root = download_dirs[0]
    records = []
    for style_name in STYLE_MAP:
        style_dir = download_root / style_name
        if not style_dir.exists():
            continue
        with os.scandir(style_dir) as entries:
            for entry in entries:
                suffix = Path(entry.name).suffix.lower()
                if suffix in IMAGE_EXTENSIONS and entry.is_file():
                    records.append(parse_download_filename(Path(entry.path)))

    records.sort(key=lambda rec: (rec['file_style_name'], rec['serial_num'], rec['filename']))
    return records


def quick_image_stats(path_text: str):
    try:
        with Image.open(path_text) as image:
            arr = np.asarray(ImageOps.exif_transpose(image).convert('L'), dtype=np.uint8)
    except Exception:
        return None
    h, w = arr.shape
    b = max(1, int(round(min(h, w) * 0.08)))
    border = np.concatenate([arr[:b, :].ravel(), arr[-b:, :].ravel(), arr[:, :b].ravel(), arr[:, -b:].ravel()])
    center = arr[int(h*0.20):max(int(h*0.80), int(h*0.20)+1), int(w*0.20):max(int(w*0.80), int(w*0.20)+1)]
    ch = max(1, int(round(h * 0.10)))
    cw = max(1, int(round(w * 0.10)))
    corners = np.concatenate([arr[:ch, :cw].ravel(), arr[:ch, -cw:].ravel(), arr[-ch:, :cw].ravel(), arr[-ch:, -cw:].ravel()])
    return {
        'mean': float(arr.mean()),
        'std': float(arr.std()),
        'border_mean': float(border.mean()),
        'corner_mean': float(corners.mean()),
        'center_mean': float(center.mean()),
        'dark_ratio': float(np.mean(arr < 80)),
        'bright_ratio': float(np.mean(arr > 200)),
        'center_dark_ratio': float(np.mean(center < 80)),
        'center_bright_ratio': float(np.mean(center > 200)),
        'mid_ratio': float(np.mean((arr >= 110) & (arr <= 220))),
    }


def make_debug_sample(meta: pd.DataFrame, sample_size: int) -> pd.DataFrame:
    selected = []
    used = set()

    quotas = [('草书', 110), ('章草', 85), ('行书', 110), ('篆书', 45), ('楷书', 45), ('隶书', 25)]
    for style_name, quota in quotas:
        part = meta[meta['style_name'] == style_name].head(quota)
        selected.append(part)
        used.update(part.index.tolist())

    # Add likely black-background and gray-background examples using quick image stats.
    special_indices = []
    candidates = meta.loc[~meta.index.isin(used)].head(5000)
    for idx, row in candidates.iterrows():
        stats = quick_image_stats(row.get('image_path', ''))
        if stats is None:
            continue
        likely_black = (
            (stats['corner_mean'] < 100 and stats['bright_ratio'] > 0.03) or
            (stats['border_mean'] < 120 and stats['bright_ratio'] > 0.03) or
            (stats['dark_ratio'] > 0.55 and stats['bright_ratio'] > 0.02) or
            (stats['corner_mean'] < stats['center_mean'] - 20 and stats['corner_mean'] < 130) or
            (stats['center_dark_ratio'] > 0.50 and stats['center_bright_ratio'] > 0.02 and stats['center_mean'] < 135)
        )
        likely_gray = stats['mid_ratio'] > 0.35 and 135 < stats['mean'] < 225
        if likely_black or likely_gray:
            special_indices.append(idx)
            if len(special_indices) >= 80:
                break
    if special_indices:
        special = meta.loc[special_indices]
        selected.append(special)
        used.update(special.index.tolist())

    current = pd.concat(selected, ignore_index=False).drop_duplicates(subset=['global_index'])
    remaining_n = max(0, sample_size - len(current))
    if remaining_n > 0:
        remaining = meta.loc[~meta.index.isin(used)].head(remaining_n)
        selected.append(remaining)

    return pd.concat(selected, ignore_index=True).drop_duplicates(subset=['global_index']).head(sample_size)



def make_profile_stratified_sample(meta: pd.DataFrame, sample_size: int) -> pd.DataFrame:
    """Prefer image_type_report.csv stratification for v2 sample tests."""
    report_path = ROOT / 'image_type_profile' / 'image_type_report.csv'
    if not report_path.exists():
        sampled = make_debug_sample(meta, sample_size).copy()
        sampled['image_type'] = 'unknown'
        return sampled

    report = pd.read_csv(report_path, encoding='utf-8-sig')
    report = report[report['style_label'].isin(STYLE_LABEL_ORDER)].copy()
    report = report[report['image_path'].map(lambda p: bool(norm_text(p)) and Path(norm_text(p)).exists())].copy()
    report = report[~report['image_type'].isin(['catalog_or_invalid'])].copy()
    report['matched_by'] = 'image_type_report'

    quotas = {
        'clean_white': 16,
        'yellow_paper': 16,
        'gray_texture': 8,
        'black_clean': 16,
        'black_gray_ink': 10,
        'rubbing_noisy': 12,
        'black_noisy': 8,
        'black_damaged': 8,
        'unknown': 6,
    }
    selected = []
    used = set()
    for image_type, quota in quotas.items():
        part = report[(report['image_type'] == image_type) & (~report.index.isin(used))]
        if not part.empty:
            take = part.sample(n=min(quota, len(part)), random_state=42 + len(selected))
            selected.append(take)
            used.update(take.index.tolist())

    current = pd.concat(selected, ignore_index=False).drop_duplicates(subset=['image_path']) if selected else pd.DataFrame(columns=report.columns)
    for style_label in STYLE_LABEL_ORDER:
        if style_label not in set(current.get('style_label', [])):
            part = report[(report['style_label'] == style_label) & (~report.index.isin(used))]
            if not part.empty:
                take = part.sample(n=1, random_state=99 + len(selected))
                selected.append(take)
                used.update(take.index.tolist())

    current = pd.concat(selected, ignore_index=False).drop_duplicates(subset=['image_path']) if selected else current
    need = sample_size - len(current)
    if need > 0:
        remaining = report[~report.index.isin(used)]
        if not remaining.empty:
            fill = remaining.sample(n=min(need, len(remaining)), random_state=141)
            current = pd.concat([current, fill], ignore_index=False).drop_duplicates(subset=['image_path'])

    sample = current.head(sample_size).copy().reset_index(drop=True)
    sample['global_index'] = range(len(sample))
    for col in ORIGINAL_COLUMNS + ['bundle', 'style_label', 'image_path', 'matched_by', 'image_type']:
        if col not in sample.columns:
            sample[col] = '' if col != 'image_type' else 'unknown'
    return sample


matched_rows = []
failure_rows = []
bundle_stats = []
global_start = 0

for bundle_name in BUNDLE_NAMES:
    df = read_bundle_csv(bundle_name, global_start)
    global_start += len(df)

    file_records = scan_bundle_image_files(bundle_name)
    indexes = make_file_indexes(file_records)
    used_paths = set()

    csv_rows = len(df)
    target_df = df[df['style_name'].isin(STYLE_MAP)].copy()
    non_target_df = df[~df['style_name'].isin(STYLE_MAP)].copy()

    for _, row in non_target_df.iterrows():
        failure_rows.append({
            'global_index': int(row['global_index']),
            'bundle': row['bundle'],
            'query_char': row.get('query_char'),
            'style_name': row.get('style_name'),
            'style_label': '',
            'author': row.get('author'),
            'work_title': row.get('work_title'),
            'image_url': row.get('image_url'),
            'image_path': '',
            'reason': 'non_target_style',
            'error': '',
        })

    found_paths = 0
    missing_paths = 0

    for _, row in target_df.iterrows():
        row_dict = row.to_dict()
        rec, matched_by = match_image_for_row(row_dict, indexes, used_paths)

        if rec is None:
            image_path = ''
            missing_paths += 1
        else:
            image_path = str(rec['path'])
            found_paths += 1

        row_dict['image_path'] = image_path
        row_dict['matched_by'] = matched_by
        matched_rows.append(row_dict)

    bundle_stats.append({
        'bundle': bundle_name,
        'csv_rows': csv_rows,
        'target_style_rows': len(target_df),
        'found_image_paths': found_paths,
        'missing_image_paths': missing_paths,
        'downloaded_target_files': len(file_records),
    })

meta_filtered = pd.DataFrame(matched_rows)
non_target_failures = pd.DataFrame(failure_rows)

if RUN_FULL:
    work_df = meta_filtered.copy()
    work_df['image_type'] = 'unknown'
else:
    work_df = make_profile_stratified_sample(meta_filtered, SAMPLE_SIZE)

print('Bundle matching summary:')
display(pd.DataFrame(bundle_stats))
print(f'Target rows selected for preprocessing: {len(work_df):,}')
print(f'Non-target rows recorded as failures: {len(non_target_failures):,}')
print('Sample style_name counts:')
display(work_df['style_name'].value_counts().to_frame('count'))


## 4. Balanced v2 preprocessing functions

`clean_background_stronger` is not used. `clean_background_balanced_v2` uses mild percentile contrast, protects dark stroke neighborhoods, whitens only high-value background, and removes tiny specks conservatively.


In [ ]:
class SkipImage(Exception):
    def __init__(self, reason, detail=''):
        super().__init__(reason)
        self.reason = reason
        self.detail = detail


def otsu_threshold(gray: np.ndarray) -> int:
    gray_u8 = np.asarray(gray, dtype=np.uint8)
    hist = np.bincount(gray_u8.ravel(), minlength=256).astype(np.float64)
    total = hist.sum()
    if total <= 0:
        return 127

    bins = np.arange(256, dtype=np.float64)
    weight_bg = np.cumsum(hist)
    weight_fg = total - weight_bg
    sum_bg = np.cumsum(hist * bins)
    sum_total = sum_bg[-1]
    valid = (weight_bg > 0) & (weight_fg > 0)

    between = np.zeros(256, dtype=np.float64)
    mean_bg = np.zeros(256, dtype=np.float64)
    mean_fg = np.zeros(256, dtype=np.float64)
    mean_bg[valid] = sum_bg[valid] / weight_bg[valid]
    mean_fg[valid] = (sum_total - sum_bg[valid]) / weight_fg[valid]
    between[valid] = weight_bg[valid] * weight_fg[valid] * (mean_bg[valid] - mean_fg[valid]) ** 2
    return int(np.argmax(between))


def connected_components(mask: np.ndarray):
    mask_u8 = mask.astype(np.uint8)
    if HAS_CV2:
        _, labels, stats, _ = cv2.connectedComponentsWithStats(mask_u8, connectivity=8)
        return labels, stats[:, cv2.CC_STAT_AREA]
    if HAS_SCIPY:
        labels, _ = ndi.label(mask_u8, structure=np.ones((3, 3), dtype=np.uint8))
        return labels, np.bincount(labels.ravel())
    return None, None


def border_pixels(gray: np.ndarray, ratio: float = 0.08) -> np.ndarray:
    h, w = gray.shape
    bh = max(1, int(round(h * ratio)))
    bw = max(1, int(round(w * ratio)))
    return np.concatenate([gray[:bh, :].ravel(), gray[-bh:, :].ravel(), gray[:, :bw].ravel(), gray[:, -bw:].ravel()])


def corner_pixels(gray: np.ndarray, ratio: float = 0.10) -> np.ndarray:
    h, w = gray.shape
    ch = max(1, int(round(h * ratio)))
    cw = max(1, int(round(w * ratio)))
    return np.concatenate([
        gray[:ch, :cw].ravel(),
        gray[:ch, -cw:].ravel(),
        gray[-ch:, :cw].ravel(),
        gray[-ch:, -cw:].ravel(),
    ])


def border_mean(gray: np.ndarray) -> float:
    return float(border_pixels(gray).mean())


def corner_mean(gray: np.ndarray) -> float:
    return float(corner_pixels(gray).mean())


def center_region(gray: np.ndarray, keep_ratio: float = 0.60) -> np.ndarray:
    h, w = gray.shape
    margin_y = max(0, int(round(h * (1.0 - keep_ratio) / 2.0)))
    margin_x = max(0, int(round(w * (1.0 - keep_ratio) / 2.0)))
    y1 = max(margin_y + 1, h - margin_y)
    x1 = max(margin_x + 1, w - margin_x)
    return gray[margin_y:y1, margin_x:x1]


def largest_component_fraction(mask: np.ndarray) -> float:
    labels, sizes = connected_components(mask)
    if labels is None or sizes is None or len(sizes) <= 1:
        return 0.0
    return float(sizes[1:].max() / max(1, mask.size))


def remove_dark_border_components(gray, dark_threshold: int = 90, min_area_fraction: float = 0.01):
    gray = np.asarray(gray, dtype=np.uint8).copy()
    mask = gray < dark_threshold
    labels, sizes = connected_components(mask)
    if labels is None or sizes is None or len(sizes) <= 1:
        return gray
    border_labels = np.unique(np.concatenate([labels[0, :], labels[-1, :], labels[:, 0], labels[:, -1]]))
    min_area = max(20, int(round(gray.size * min_area_fraction)))
    for label in border_labels:
        if label != 0 and sizes[label] >= min_area:
            gray[labels == label] = 255
    return gray


def inversion_stats(gray: np.ndarray) -> dict:
    gray = np.asarray(gray, dtype=np.uint8)
    center = center_region(gray, keep_ratio=0.60)
    stats = {
        'border_mean': float(border_pixels(gray, ratio=0.08).mean()),
        'corner_mean': float(corner_pixels(gray, ratio=0.10).mean()),
        'dark_ratio': float(np.mean(gray < 80)),
        'bright_ratio': float(np.mean(gray > 200)),
        'center_mean': float(center.mean()),
        'center_dark_ratio': float(np.mean(center < 80)),
        'center_bright_ratio': float(np.mean(center > 200)),
    }
    return stats


def is_black_background_or_white_ink(gray) -> bool:
    """
    Return True if the image is likely black background with white ink.
    Uses border, corner, global foreground, and center statistics.
    """
    stats = inversion_stats(gray)
    border_m = stats['border_mean']
    corner_m = stats['corner_mean']
    dark_ratio = stats['dark_ratio']
    bright_ratio = stats['bright_ratio']
    center_m = stats['center_mean']
    center_dark_ratio = stats['center_dark_ratio']
    center_bright_ratio = stats['center_bright_ratio']

    rule_a = corner_m < 100 and bright_ratio > 0.03
    rule_b = border_m < 120 and bright_ratio > 0.03
    rule_c = dark_ratio > 0.55 and bright_ratio > 0.02
    rule_d = corner_m < center_m - 20 and corner_m < 130

    # Catches black-background glyphs with a white outer frame: corners/borders can look white,
    # but the central character area is still mostly dark with bright strokes.
    rule_e = center_dark_ratio > 0.50 and center_bright_ratio > 0.02 and center_m < 135
    return bool(rule_a or rule_b or rule_c or rule_d or rule_e)


def white_background_score(gray) -> float:
    gray = np.asarray(gray, dtype=np.uint8)
    border = border_pixels(gray, ratio=0.08)
    corners = corner_pixels(gray, ratio=0.10)
    center = center_region(gray, keep_ratio=0.60)
    p90 = float(np.percentile(gray, 90))
    p95 = float(np.percentile(gray, 95))
    return float(corners.mean()) * 0.30 + float(border.mean()) * 0.25 + p90 * 0.20 + p95 * 0.15 + float(center.mean()) * 0.10


def ensure_white_background_black_ink(gray):
    """
    Ensure image is white background with black ink.
    Black-background / white-ink images are inverted before cleanup.
    """
    gray = np.asarray(gray, dtype=np.uint8)

    if is_black_background_or_white_ink(gray):
        gray = 255 - gray

    if is_black_background_or_white_ink(gray):
        candidate = 255 - gray
        if (not is_black_background_or_white_ink(candidate)) and white_background_score(candidate) > white_background_score(gray):
            gray = candidate
        else:
            raise ValueError('black_background_after_processing')

    return gray.astype(np.uint8)


def is_black_background(img) -> bool:
    arr = np.asarray(img, dtype=np.uint8)
    return is_black_background_or_white_ink(arr) or border_mean(arr) < 150


def percentile_normalize_light(gray: np.ndarray) -> np.ndarray:
    gray = np.asarray(gray, dtype=np.uint8)
    arr = gray.astype(np.float32)
    low = np.percentile(arr, 1)
    high = np.percentile(arr, 98.5)
    if high <= low + 1e-6:
        return gray.copy()
    arr = (arr - low) / (high - low + 1e-6) * 255.0
    return np.clip(arr, 0, 255).astype(np.uint8)


def mask_without_tiny_components(mask: np.ndarray, min_area: int = BBOX_MIN_COMPONENT_AREA) -> np.ndarray:
    labels, sizes = connected_components(mask)
    if labels is None or sizes is None or len(sizes) <= 1:
        return mask
    keep = np.where((np.arange(len(sizes)) != 0) & (sizes >= min_area))[0]
    if len(keep) == 0:
        return mask
    return np.isin(labels, keep)


def bbox_from_mask(mask: np.ndarray):
    ys, xs = np.where(mask)
    if len(xs) == 0 or len(ys) == 0:
        return None
    return int(xs.min()), int(ys.min()), int(xs.max()) + 1, int(ys.max()) + 1


def bbox_for_speck_removal(gray: np.ndarray):
    mask = mask_without_tiny_components(gray < 180, min_area=20)
    return bbox_from_mask(mask)


def remove_tiny_specks(gray, inside_min_area: int = TINY_SPECK_AREA, outside_min_area: int = OUTSIDE_BBOX_SPECK_AREA, dark_threshold: int = 90):
    gray = np.asarray(gray, dtype=np.uint8).copy()
    dark_mask = gray < dark_threshold
    labels, sizes = connected_components(dark_mask)
    if labels is None or sizes is None or len(sizes) <= 1:
        return gray

    bbox = bbox_for_speck_removal(gray)
    inside_bbox = np.zeros_like(dark_mask, dtype=bool)
    if bbox is not None:
        x0, y0, x1, y1 = bbox
        inside_bbox[y0:y1, x0:x1] = True

    for label in range(1, len(sizes)):
        area = sizes[label]
        component = labels == label
        is_inside = bool(np.any(component & inside_bbox))
        threshold = inside_min_area if is_inside else outside_min_area
        if area < threshold:
            gray[component] = 255
    return gray


def dilate_mask_once(mask: np.ndarray) -> np.ndarray:
    if HAS_CV2:
        return cv2.dilate(mask.astype(np.uint8), np.ones((2, 2), np.uint8), iterations=1).astype(bool)
    if HAS_SCIPY:
        return ndi.binary_dilation(mask, structure=np.ones((2, 2), dtype=bool), iterations=1)
    return mask


def clean_background_balanced_v2(gray):
    gray = ensure_white_background_black_ink(gray)
    gray = percentile_normalize_light(gray)
    gray = ensure_white_background_black_ink(gray)

    cleaned = gray.copy()
    background_threshold = max(215, int(np.percentile(gray, 82)))
    dark_stroke_mask = gray < 160
    protected_mask = dilate_mask_once(dark_stroke_mask)

    very_bright = gray > 235
    light_background = (gray > background_threshold) & (gray > 180) & (~protected_mask)
    cleaned[very_bright | light_background] = 255

    cleaned = remove_tiny_specks(cleaned, inside_min_area=8, outside_min_area=25, dark_threshold=90)
    cleaned = ensure_white_background_black_ink(cleaned)
    cleaned[cleaned > 242] = 255
    return cleaned.astype(np.uint8)


def crop_mask(gray: np.ndarray) -> np.ndarray:
    mask = mask_without_tiny_components(gray < 210, min_area=BBOX_MIN_COMPONENT_AREA)
    if float(mask.mean()) < 0.002:
        mask = mask_without_tiny_components(gray < 230, min_area=BBOX_MIN_COMPONENT_AREA)
    return mask


def validate_single_character_image(gray: np.ndarray):
    h, w = gray.shape
    aspect = max(w / max(h, 1), h / max(w, 1))
    if aspect > 2.5:
        raise SkipImage('suspicious_aspect_ratio', f'image aspect={aspect:.3f}')

    mask = crop_mask(gray)
    bbox = bbox_from_mask(mask)
    if bbox is None:
        raise SkipImage('invalid_content_bbox', 'no foreground bbox')

    x0, y0, x1, y1 = bbox
    bbox_area_fraction = ((x1 - x0) * (y1 - y0)) / max(1, h * w)
    if bbox_area_fraction < 0.03:
        raise SkipImage('invalid_content_bbox', f'bbox_area_fraction={bbox_area_fraction:.6f}')

    bbox_aspect = max((x1 - x0) / max(1, y1 - y0), (y1 - y0) / max(1, x1 - x0))
    if bbox_aspect > 2.8:
        raise SkipImage('suspicious_aspect_ratio', f'bbox aspect={bbox_aspect:.3f}')
    return bbox


def crop_pad_resize(gray: np.ndarray, bbox, target_size: int = TARGET_SIZE) -> Image.Image:
    h, w = gray.shape
    x0, y0, x1, y1 = bbox
    box_size = max(x1 - x0, y1 - y0)
    margin = max(4, int(round(box_size * 0.20)))
    x0 = max(0, x0 - margin)
    y0 = max(0, y0 - margin)
    x1 = min(w, x1 + margin)
    y1 = min(h, y1 + margin)
    crop = gray[y0:y1, x0:x1]
    if crop.size == 0:
        raise SkipImage('invalid_content_bbox', 'empty crop')

    ch, cw = crop.shape
    side = max(ch, cw)
    square = np.full((side, side), 255, dtype=np.uint8)
    y_offset = (side - ch) // 2
    x_offset = (side - cw) // 2
    square[y_offset:y_offset + ch, x_offset:x_offset + cw] = crop
    return Image.fromarray(square, mode='L').resize((target_size, target_size), Image.Resampling.LANCZOS)


def validate_processed_quality(processed):
    arr = np.asarray(processed, dtype=np.uint8)
    if arr.shape != (TARGET_SIZE, TARGET_SIZE):
        raise ValueError(f'invalid_output_size={arr.shape}')
    if is_black_background_or_white_ink(arr) or border_mean(arr) < 150:
        raise ValueError('black_background_after_processing')

    foreground_ratio = float(np.mean(arr < 220))
    if foreground_ratio < 0.006:
        raise ValueError('too_little_ink_after_processing')
    if foreground_ratio > 0.60:
        raise ValueError('too_much_ink_after_processing')

    dark_mask = arr < 160
    labels, sizes = connected_components(dark_mask)
    bbox = bbox_from_mask(crop_mask(arr))
    if bbox is None:
        raise ValueError('invalid_content_bbox')
    x0, y0, x1, y1 = bbox
    bbox_area_fraction = ((x1 - x0) * (y1 - y0)) / max(1, arr.size)
    if bbox_area_fraction < 0.03:
        raise ValueError('invalid_content_bbox')

    if sizes is not None and len(sizes) > 1:
        valid_sizes = sizes[1:]
        component_count = int(np.sum(valid_sizes >= 4))
        if component_count > 40:
            raise ValueError('too_noisy_or_fragmented')
        total_dark_area = int(valid_sizes.sum())
        largest = int(valid_sizes.max())
        if total_dark_area > 0 and largest / total_dark_area < 0.08 and component_count > 15:
            raise ValueError('fragmented_character')


def finalize_processed_image(image: Image.Image) -> Image.Image:
    arr = np.asarray(image.convert('L'), dtype=np.uint8)
    arr = ensure_white_background_black_ink(arr)
    arr[arr > 242] = 255

    if is_black_background_or_white_ink(arr):
        raise ValueError('black_background_after_processing')

    validate_processed_quality(arr)
    return Image.fromarray(arr.astype(np.uint8), mode='L')



def force_invert_black_background_source(gray):
    """
    Special-path orientation for rubbings / black-background white-ink images.
    This is intentionally separate from the normal pipeline so normal images are not affected.
    """
    gray = np.asarray(gray, dtype=np.uint8)
    inverted = 255 - gray

    if is_black_background_or_white_ink(gray):
        gray = inverted
    elif white_background_score(inverted) > white_background_score(gray) + 6:
        gray = inverted

    # White outer frames on black-background originals become dark borders after inversion.
    # Remove only large dark components touching the image edge before bbox detection.
    gray = remove_dark_border_components(gray, dark_threshold=105, min_area_fraction=0.004)
    return gray.astype(np.uint8)


def percentile_normalize_black_bg(gray: np.ndarray) -> np.ndarray:
    gray = np.asarray(gray, dtype=np.uint8)
    arr = gray.astype(np.float32)
    low = np.percentile(arr, 1)
    high = np.percentile(arr, 99)
    if high <= low + 1e-6:
        return gray.copy()
    arr = (arr - low) / (high - low + 1e-6) * 255.0
    return np.clip(arr, 0, 255).astype(np.uint8)


def black_bg_special_cleanup(gray):
    gray = force_invert_black_background_source(gray)
    gray = percentile_normalize_black_bg(gray)
    gray = remove_dark_border_components(gray, dark_threshold=105, min_area_fraction=0.004)

    cleaned = gray.copy()
    cleaned[cleaned > 225] = 255
    # Preserve gray ink texture. Only remove tiny isolated dark specks, with stricter cleanup outside the main bbox.
    cleaned = remove_tiny_specks(cleaned, inside_min_area=8, outside_min_area=25, dark_threshold=95)
    cleaned[cleaned > 238] = 255
    return cleaned.astype(np.uint8)


def black_bg_special_bbox(gray):
    mask = mask_without_tiny_components(gray < 210, min_area=8)
    if float(mask.mean()) < 0.002:
        mask = mask_without_tiny_components(gray < 230, min_area=8)
    bbox = bbox_from_mask(mask)
    if bbox is None:
        raise ValueError('black_bg_special_failed: invalid_content_bbox')

    h, w = gray.shape
    x0, y0, x1, y1 = bbox
    bbox_area_fraction = ((x1 - x0) * (y1 - y0)) / max(1, h * w)
    if bbox_area_fraction < 0.02:
        raise ValueError('black_bg_special_failed: invalid_content_bbox')
    bbox_aspect = max((x1 - x0) / max(1, y1 - y0), (y1 - y0) / max(1, x1 - x0))
    if bbox_aspect > 3.2:
        raise ValueError('black_bg_special_failed: suspicious_aspect_ratio')
    return bbox


def validate_black_bg_special_quality(processed):
    arr = np.asarray(processed, dtype=np.uint8)
    if arr.shape != (TARGET_SIZE, TARGET_SIZE):
        raise ValueError(f'black_bg_special_failed: invalid_output_size={arr.shape}')
    if is_black_background_or_white_ink(arr) or border_mean(arr) < 150 or corner_mean(arr) < 140:
        raise ValueError('black_bg_special_failed: black_background_after_processing')

    foreground_ratio = float(np.mean(arr < 220))
    if foreground_ratio < 0.006:
        raise ValueError('black_bg_special_failed: too_little_ink_after_processing')
    if foreground_ratio > 0.65:
        raise ValueError('black_bg_special_failed: too_much_ink_after_processing')

    dark_mask = arr < 160
    labels, sizes = connected_components(dark_mask)
    if sizes is not None and len(sizes) > 1:
        valid_sizes = sizes[1:]
        component_count = int(np.sum(valid_sizes >= 4))
        if component_count > 55:
            raise ValueError('black_bg_special_failed: too_noisy_or_fragmented')
        total_dark_area = int(valid_sizes.sum())
        largest = int(valid_sizes.max())
        if total_dark_area > 0 and largest / total_dark_area < 0.06 and component_count > 20:
            raise ValueError('black_bg_special_failed: fragmented_character')


def preprocess_black_background_image(path: Path) -> Image.Image:
    """
    Dedicated fallback for black-background / white-ink rubbing images.
    Normal preprocessing remains unchanged; this runs only after normal mode fails as black background.
    """
    try:
        with Image.open(path) as image:
            image = ImageOps.exif_transpose(image).convert('L')
            gray = np.asarray(image, dtype=np.uint8)
    except (UnidentifiedImageError, OSError, FileNotFoundError) as exc:
        raise SkipImage('image_read_failed', repr(exc)) from exc

    cleaned = black_bg_special_cleanup(gray)
    bbox = black_bg_special_bbox(cleaned)
    processed = crop_pad_resize(cleaned, bbox, target_size=TARGET_SIZE)
    arr = np.asarray(processed.convert('L'), dtype=np.uint8)
    arr = force_invert_black_background_source(arr)
    arr[arr > 238] = 255

    if is_black_background_or_white_ink(arr):
        raise ValueError('black_bg_special_failed: black_background_after_processing')

    validate_black_bg_special_quality(arr)
    return Image.fromarray(arr.astype(np.uint8), mode='L')



def binary_dilate(mask: np.ndarray, iterations: int = 1) -> np.ndarray:
    out = mask.astype(bool)
    for _ in range(iterations):
        padded = np.pad(out, 1, mode='constant', constant_values=False)
        out = (
            padded[:-2, :-2] | padded[:-2, 1:-1] | padded[:-2, 2:] |
            padded[1:-1, :-2] | padded[1:-1, 1:-1] | padded[1:-1, 2:] |
            padded[2:, :-2] | padded[2:, 1:-1] | padded[2:, 2:]
        )
    return out


def remove_bottom_right_watermark(processed):
    """Clear small, thin bottom-right watermark components without blanking the glyph."""
    arr = np.asarray(processed.convert('L') if isinstance(processed, Image.Image) else processed, dtype=np.uint8).copy()
    h, w = arr.shape
    roi_x0 = int(round(w * 0.50))
    roi_y0 = int(round(h * 0.62))
    darkish = arr < 218
    labels, sizes = connected_components(darkish)
    if labels is None or sizes is None or len(sizes) <= 1:
        return Image.fromarray(arr, mode='L')

    remove = np.zeros_like(darkish, dtype=bool)
    image_area = h * w
    for label in range(1, len(sizes)):
        component = labels == label
        ys, xs = np.where(component)
        if len(xs) == 0:
            continue
        x0, x1 = int(xs.min()), int(xs.max()) + 1
        y0, y1 = int(ys.min()), int(ys.max()) + 1
        area = int(sizes[label])
        bw = x1 - x0
        bh = y1 - y0
        centroid_x = float(xs.mean())
        centroid_y = float(ys.mean())
        if centroid_x < roi_x0 or centroid_y < roi_y0:
            continue

        horizontal_thin = bw >= 10 and bh <= 9 and bw / max(1, bh) >= 2.2
        small_speck = area <= max(16, int(image_area * 0.0025))
        small_text_piece = area <= max(60, int(image_area * 0.006)) and (bw <= 42 or bh <= 14)
        far_corner_piece = x0 >= int(w * 0.62) and y0 >= int(h * 0.72) and area <= max(110, int(image_area * 0.010))
        if horizontal_thin or small_speck or small_text_piece or far_corner_piece:
            remove |= component

    if remove.any():
        remove = binary_dilate(remove, iterations=1)
        remove[:roi_y0, :] = False
        remove[:, :roi_x0] = False
        arr[remove] = 255
    return Image.fromarray(arr, mode='L')


def whiten_background_final(processed):
    """Push background texture/noise toward white while preserving ink and edges."""
    arr = np.asarray(processed.convert('L') if isinstance(processed, Image.Image) else processed, dtype=np.uint8).astype(np.float32)
    ink_core = arr < 175
    protected = binary_dilate(ink_core, iterations=1)
    out = arr.copy()
    strong_bg = (out >= 232) & ~protected
    mid_bg = (out >= 205) & (out < 232) & ~protected
    light_noise = (out >= 185) & (out < 205) & ~protected
    out[strong_bg] = 255
    out[mid_bg] = out[mid_bg] + (255 - out[mid_bg]) * 0.82
    out[light_noise] = out[light_noise] + (255 - out[light_noise]) * 0.45
    return Image.fromarray(np.clip(out, 0, 255).astype(np.uint8), mode='L')


def background_dirty_ratio(processed) -> float:
    arr = np.asarray(processed.convert('L') if isinstance(processed, Image.Image) else processed, dtype=np.uint8)
    ink_core = arr < 175
    protected = binary_dilate(ink_core, iterations=1)
    dirty = (arr >= 185) & (arr < 250) & ~protected
    return float(np.mean(dirty))


def apply_final_v2_postprocessing(processed):
    arr = np.asarray(processed.convert('L') if isinstance(processed, Image.Image) else processed, dtype=np.uint8)
    arr = ensure_white_background_black_ink(arr)
    img = Image.fromarray(arr.astype(np.uint8), mode='L')
    before_wm = np.asarray(img, dtype=np.uint8)
    after_wm_img = remove_bottom_right_watermark(img)
    after_wm = np.asarray(after_wm_img, dtype=np.uint8)
    watermark_removed = bool(np.any(before_wm != after_wm))
    watermark_attempted = watermark_removed or bool(np.mean(before_wm[int(before_wm.shape[0]*0.62):, int(before_wm.shape[1]*0.50):] < 230) > 0.003)
    img = whiten_background_final(after_wm_img)
    arr = ensure_white_background_black_ink(np.asarray(img, dtype=np.uint8))
    img = Image.fromarray(arr.astype(np.uint8), mode='L')
    validate_processed_quality(np.asarray(img, dtype=np.uint8))
    return img, {
        'watermark_attempted': bool(watermark_attempted),
        'watermark_removed': bool(watermark_removed),
        'background_dirty_ratio': background_dirty_ratio(img),
    }


def preprocess_image(path: Path) -> Image.Image:
    try:
        with Image.open(path) as image:
            image = ImageOps.exif_transpose(image).convert('L')
            gray = np.asarray(image, dtype=np.uint8)
    except (UnidentifiedImageError, OSError, FileNotFoundError) as exc:
        raise SkipImage('image_read_failed', repr(exc)) from exc

    gray = ensure_white_background_black_ink(gray)
    cleaned = clean_background_balanced_v2(gray)
    bbox = validate_single_character_image(cleaned)
    processed = crop_pad_resize(cleaned, bbox, target_size=TARGET_SIZE)
    return finalize_processed_image(processed)


## 5. Run sample preprocessing

This writes successful 128x128 PNGs for the 100-row v2 debug sample and records skipped/failed rows.


In [ ]:
success_records = []
failure_records = []
output_image_root_resolved = OUTPUT_IMAGE_ROOT.resolve()

KNOWN_FAILURE_REASONS = [
    'black_background_after_processing',
    'black_bg_special_failed',
    'too_little_ink_after_processing',
    'too_much_ink_after_processing',
    'too_noisy_or_fragmented',
    'fragmented_character',
    'invalid_content_bbox',
]


def reason_from_exception(exc):
    message = str(exc)
    for item in KNOWN_FAILURE_REASONS:
        if item in message:
            return item
    return 'processing_error'


for _, row in tqdm(work_df.iterrows(), total=len(work_df), desc='Preprocessing target rows', mininterval=5):
    row_dict = row.to_dict()
    global_index = int(row_dict['global_index'])
    image_path_text = norm_text(row_dict.get('image_path'))
    matched_by = row_dict.get('matched_by', '')

    base_failure = {
        'global_index': global_index,
        'bundle': row_dict.get('bundle'),
        'query_char': row_dict.get('query_char'),
        'style_name': row_dict.get('style_name'),
        'style_label': row_dict.get('style_label'),
        'author': row_dict.get('author'),
        'work_title': row_dict.get('work_title'),
        'image_url': row_dict.get('image_url'),
        'image_path': image_path_text,
        'reason': '',
        'error': '',
    }

    metadata_has_hint = contains_vector_hint(*row_text_values(row_dict))
    filename_has_hint = contains_vector_hint(image_path_text)
    if metadata_has_hint or filename_has_hint:
        base_failure['reason'] = 'vector_hint_image'
        failure_records.append(base_failure)
        continue

    if not image_path_text:
        base_failure['reason'] = 'no_image_found'
        failure_records.append(base_failure)
        continue

    source_path = Path(image_path_text)
    if not source_path.exists():
        base_failure['reason'] = 'no_image_found'
        base_failure['error'] = 'matched image path does not exist'
        failure_records.append(base_failure)
        continue

    processed_path = (OUTPUT_IMAGE_ROOT / row_dict['style_label'] / output_filename(row_dict)).resolve()
    if output_image_root_resolved not in processed_path.parents:
        base_failure['reason'] = 'processing_error'
        base_failure['error'] = f'processed_path escaped output root: {processed_path}'
        failure_records.append(base_failure)
        continue

    processing_mode = 'normal'
    normal_error = None

    try:
        processed = preprocess_image(source_path)
    except SkipImage as exc:
        base_failure['reason'] = exc.reason
        base_failure['error'] = exc.detail
        failure_records.append(base_failure)
        continue
    except Exception as exc:
        normal_error = exc
        reason = reason_from_exception(exc)
        if reason != 'black_background_after_processing':
            base_failure['reason'] = reason
            base_failure['error'] = repr(exc) + '\n' + traceback.format_exc(limit=2)
            failure_records.append(base_failure)
            continue

        try:
            processed = preprocess_black_background_image(source_path)
            processing_mode = 'black_bg_special'
        except SkipImage as exc2:
            base_failure['reason'] = 'black_bg_special_failed'
            base_failure['error'] = f'normal_error={repr(normal_error)}\nspecial_skip={exc2.reason}: {exc2.detail}'
            failure_records.append(base_failure)
            continue
        except Exception as exc2:
            base_failure['reason'] = 'black_bg_special_failed'
            base_failure['error'] = (
                f'normal_error={repr(normal_error)}\n'
                f'special_error={repr(exc2)}\n'
                f'{traceback.format_exc(limit=2)}'
            )
            failure_records.append(base_failure)
            continue

    try:
        processed, post_stats = apply_final_v2_postprocessing(processed)
        processed_arr = np.asarray(processed, dtype=np.uint8)
        if processed.size != (TARGET_SIZE, TARGET_SIZE):
            raise ValueError(f'invalid_output_size={processed.size}')
        validate_processed_quality(processed_arr)
        processed.save(processed_path, format='PNG')

        success_records.append({
            'global_index': global_index,
            'bundle': row_dict.get('bundle'),
            'query_char': row_dict.get('query_char'),
            'style_code': row_dict.get('style_code'),
            'style_name': row_dict.get('style_name'),
            'style_label': row_dict.get('style_label'),
            'era': row_dict.get('era'),
            'author': row_dict.get('author'),
            'work_title': row_dict.get('work_title'),
            'detail_title': row_dict.get('detail_title'),
            'image_url': row_dict.get('image_url'),
            'thumb_url': row_dict.get('thumb_url'),
            'image_path': str(source_path),
            'processed_path': str(processed_path),
            'matched_by': matched_by,
            'image_type': row_dict.get('image_type', 'unknown'),
            'processing_mode': processing_mode,
            'background_dirty_ratio': post_stats.get('background_dirty_ratio'),
            'watermark_attempted': post_stats.get('watermark_attempted'),
            'watermark_removed': post_stats.get('watermark_removed'),
        })
    except Exception as exc:
        reason = reason_from_exception(exc)
        if processing_mode == 'black_bg_special':
            reason = 'black_bg_special_failed'
        base_failure['reason'] = reason
        base_failure['error'] = repr(exc) + '\n' + traceback.format_exc(limit=2)
        failure_records.append(base_failure)

success_df = pd.DataFrame(success_records)
target_failure_df = pd.DataFrame(failure_records)

failure_columns = [
    'global_index', 'bundle', 'query_char', 'style_name', 'style_label',
    'author', 'work_title', 'image_url', 'image_path', 'image_type', 'processing_mode', 'reason', 'error'
]
# v2 sample runs only report failures for selected target rows.
all_failures_df = target_failure_df.reindex(columns=failure_columns)

metadata_columns = [
    'global_index', 'bundle', 'query_char', 'style_code', 'style_name', 'style_label',
    'era', 'author', 'work_title', 'detail_title', 'image_url', 'thumb_url',
    'image_path', 'processed_path', 'matched_by', 'image_type', 'processing_mode',
    'background_dirty_ratio', 'watermark_attempted', 'watermark_removed'
]
success_df = success_df.reindex(columns=metadata_columns)

metadata_path = OUTPUT_ROOT / 'metadata.csv'
failures_path = OUTPUT_ROOT / 'preprocessing_failures.csv'
success_df.to_csv(metadata_path, index=False, encoding='utf-8-sig')
all_failures_df.to_csv(failures_path, index=False, encoding='utf-8-sig')

print(f'Metadata written to: {metadata_path}')
print(f'Failures written to: {failures_path}')


## 6. Final validation and summary

Validate processed files for 128x128 size, white-background border brightness, foreground ratio, and quality failure counts.


In [ ]:
processed_exists = success_df['processed_path'].map(lambda p: Path(p).exists()) if not success_df.empty else pd.Series(dtype=bool)
missing_processed_paths = success_df.loc[~processed_exists, ['global_index', 'processed_path']] if not success_df.empty else pd.DataFrame()

bundle_success_counts = success_df['bundle'].value_counts().to_dict() if not success_df.empty else {}
bundle_failure_counts = target_failure_df['bundle'].value_counts().to_dict() if not target_failure_df.empty else {}

bundle_summary = pd.DataFrame(bundle_stats)
bundle_summary['success_outputs'] = bundle_summary['bundle'].map(bundle_success_counts).fillna(0).astype(int)
bundle_summary['target_failures_or_skips'] = bundle_summary['bundle'].map(bundle_failure_counts).fillna(0).astype(int)

style_success_counts = success_df['style_label'].value_counts().reindex(STYLE_LABEL_ORDER).fillna(0).astype(int)
failure_reason_counts = all_failures_df['reason'].value_counts() if not all_failures_df.empty else pd.Series(dtype='int64')
processing_mode_counts = success_df['processing_mode'].value_counts() if 'processing_mode' in success_df.columns and not success_df.empty else pd.Series(dtype='int64')
black_bg_special_success = int(processing_mode_counts.get('black_bg_special', 0))
black_bg_special_failed = int(failure_reason_counts.get('black_bg_special_failed', 0))
black_bg_special_attempted = black_bg_special_success + black_bg_special_failed

validation_bad_rows = []
for path_text in success_df['processed_path'].tolist() if not success_df.empty else []:
    path = Path(path_text)
    try:
        with Image.open(path) as image:
            arr = np.asarray(image.convert('L'), dtype=np.uint8)
            fg_ratio = float(np.mean(arr < 220))
            if image.size != (TARGET_SIZE, TARGET_SIZE):
                validation_bad_rows.append({'processed_path': path_text, 'issue': f'size={image.size}'})
            elif is_black_background_or_white_ink(arr) or border_mean(arr) <= 150:
                validation_bad_rows.append({'processed_path': path_text, 'issue': f'black_or_border_mean={border_mean(arr):.2f}'})
            elif fg_ratio < 0.006 or fg_ratio > 0.60:
                validation_bad_rows.append({'processed_path': path_text, 'issue': f'foreground_ratio={fg_ratio:.6f}'})
    except Exception as exc:
        validation_bad_rows.append({'processed_path': path_text, 'issue': repr(exc)})

validation_bad_df = pd.DataFrame(validation_bad_rows)

print(f'Output root: {OUTPUT_ROOT}')
print(f'Processed image root: {OUTPUT_IMAGE_ROOT}')
print(f'Total target rows processed: {len(work_df):,}')
print(f'Successful processed images: {len(success_df):,}')
print(f'Target failures / skips: {len(target_failure_df):,}')
print(f'All failure rows including non-target styles: {len(all_failures_df):,}')
print(f'metadata.csv rows: {len(success_df):,}')
print(f'preprocessing_failures.csv rows: {len(all_failures_df):,}')
print(f'All processed_path files exist: {missing_processed_paths.empty}')
print(f'normal success count: {int(processing_mode_counts.get("normal", 0))}')
print(f'black_bg_special attempted: {black_bg_special_attempted}')
print(f'black_bg_special success: {black_bg_special_success}')
print(f'black_bg_special failed: {black_bg_special_failed}')
print(f'black_background_after_processing count: {int(failure_reason_counts.get("black_background_after_processing", 0))}')
print(f'too_little_ink_after_processing count: {int(failure_reason_counts.get("too_little_ink_after_processing", 0))}')
print(f'too_noisy_or_fragmented count: {int(failure_reason_counts.get("too_noisy_or_fragmented", 0))}')
print(f'fragmented_character count: {int(failure_reason_counts.get("fragmented_character", 0))}')
print(f'All processed samples pass size/border/foreground checks: {validation_bad_df.empty}')

print('\nPer-bundle summary:')
display(bundle_summary)
print('\nStyle success counts:')
display(style_success_counts.to_frame('success_count'))
print('\nProcessing mode counts:')
display(processing_mode_counts.to_frame('success_count'))
print('\nFailure reason counts:')
display(failure_reason_counts.to_frame('count'))

if not validation_bad_df.empty:
    print('\nValidation issues:')
    display(validation_bad_df.head(50))


## 7. Before / after comparison and skipped examples

Randomly show 30 processed examples, additional 草書 / 章草 / 行書 examples, and 20 skipped target images with reasons.


In [ ]:
def display_figure(fig):
    try:
        from IPython.display import display
        display(fig)
    finally:
        plt.close(fig)


def show_before_after(df: pd.DataFrame, n: int = 20, title: str = 'Before / After'):
    if df.empty:
        print(f'No rows for {title}')
        return
    sample = df.sample(n=min(n, len(df)), random_state=42).reset_index(drop=True)
    fig, axes = plt.subplots(len(sample), 2, figsize=(7, max(2.5, 2.4 * len(sample))))
    if len(sample) == 1:
        axes = np.array([axes])
    fig.suptitle(title, y=1.0)
    for i, row in sample.iterrows():
        before = Image.open(row['image_path']).convert('L')
        after = Image.open(row['processed_path']).convert('L')
        axes[i, 0].imshow(before, cmap='gray', vmin=0, vmax=255)
        axes[i, 0].set_title(f"Before {row['query_char']} {row['style_name']}")
        axes[i, 0].axis('off')
        axes[i, 1].imshow(after, cmap='gray', vmin=0, vmax=255)
        axes[i, 1].set_title(f"After {row['style_label']}")
        axes[i, 1].axis('off')
    plt.tight_layout()
    display_figure(fig)


def show_skipped(df: pd.DataFrame, n: int = 20, title: str = 'Skipped target images'):
    target_skips = df[(df['reason'] != 'non_target_style') & df['image_path'].map(lambda x: bool(norm_text(x)) and Path(norm_text(x)).exists())]
    if target_skips.empty:
        print('No skipped target images with readable paths.')
        return
    sample = target_skips.sample(n=min(n, len(target_skips)), random_state=7).reset_index(drop=True)
    fig, axes = plt.subplots(len(sample), 1, figsize=(4.5, max(2.5, 2.2 * len(sample))))
    if len(sample) == 1:
        axes = np.array([axes])
    fig.suptitle(title, y=1.0)
    for i, row in sample.iterrows():
        img = Image.open(row['image_path']).convert('L')
        axes[i].imshow(img, cmap='gray', vmin=0, vmax=255)
        axes[i].set_title(f"{row.get('query_char', '')} {row.get('style_name', '')} | {row['reason']}")
        axes[i].axis('off')
    plt.tight_layout()
    display_figure(fig)

show_before_after(success_df, n=30, title='Random 30 before / after')
fragile = success_df[success_df['style_name'].isin(['草书', '章草', '行书'])]
show_before_after(fragile, n=16, title='草書 / 章草 / 行書 before / after')

suspect_black_indices = []
for idx, row in success_df.iterrows():
    try:
        with Image.open(row['image_path']) as image:
            raw = np.asarray(ImageOps.exif_transpose(image).convert('L'), dtype=np.uint8)
        if is_black_background_or_white_ink(raw):
            suspect_black_indices.append(idx)
    except Exception:
        pass
suspect_black_df = success_df.loc[suspect_black_indices]
show_before_after(suspect_black_df, n=20, title='Suspected black-background originals before / after')
show_skipped(all_failures_df, n=20, title='Skipped target images with reason')


def save_before_after_sheet(df: pd.DataFrame, out_path: Path, n: int = 30, title: str = 'Before / After'):
    if df.empty:
        print(f'No rows for {title}; not writing {out_path}')
        return
    sample = df.sample(n=min(n, len(df)), random_state=11).reset_index(drop=True)
    cell_w, cell_h = 280, 180
    width = cell_w * 2
    height = 44 + cell_h * len(sample)
    sheet = Image.new('RGB', (width, height), 'white')
    draw = ImageDraw.Draw(sheet)
    draw.text((10, 12), title, fill='black')

    for i, row in sample.iterrows():
        y = 44 + i * cell_h
        before = Image.open(row['image_path']).convert('L')
        after = Image.open(row['processed_path']).convert('L')
        before.thumbnail((128, 128), Image.Resampling.LANCZOS)
        after.thumbnail((128, 128), Image.Resampling.LANCZOS)
        before_canvas = Image.new('L', (128, 128), 255)
        after_canvas = Image.new('L', (128, 128), 255)
        before_canvas.paste(before, ((128 - before.width) // 2, (128 - before.height) // 2))
        after_canvas.paste(after, ((128 - after.width) // 2, (128 - after.height) // 2))
        sheet.paste(before_canvas.convert('RGB'), (10, y + 28))
        sheet.paste(after_canvas.convert('RGB'), (cell_w + 10, y + 28))
        author = norm_text(row.get('author')) or 'unknown_author'
        label = f"{row.get('query_char', '')} {row.get('style_label', '')} {author}"
        draw.text((10, y + 6), label[:38], fill='black')
        draw.text((cell_w + 10, y + 6), 'black_bg_special', fill='black')

    sheet.save(out_path)
    print(f'Wrote debug sheet: {out_path}')

black_bg_special_df = success_df[success_df.get('processing_mode', pd.Series(dtype=str)).eq('black_bg_special')] if not success_df.empty else pd.DataFrame()
show_before_after(black_bg_special_df, n=30, title='black_bg_special successful before / after')
save_before_after_sheet(black_bg_special_df, OUTPUT_ROOT / 'debug_black_bg_special_before_after.png', n=30, title='black_bg_special before / after')



def save_v2_debug_sheet(df: pd.DataFrame, out_path: Path, n: int = 100, watermark_only: bool = False):
    rows = df[df.get('watermark_attempted', pd.Series(dtype=bool)).astype(bool)].copy() if watermark_only else df.copy()
    rows = rows.head(n)
    cell_w, cell_h = 272, 220
    cols = 4
    row_count = max(1, math.ceil(len(rows) / cols))
    sheet = Image.new('RGB', (cols * cell_w * 2, row_count * cell_h + 36), 'white')
    draw = ImageDraw.Draw(sheet)
    draw.text((12, 10), 'watermark cases before / after' if watermark_only else 'test100 v2 before / after', fill='black')
    if rows.empty:
        draw.text((12, 52), 'No rows', fill='black')
        sheet.save(out_path)
        print(f'Wrote debug sheet: {out_path}')
        return
    for pos, (_, row) in enumerate(rows.iterrows()):
        block_x = (pos % cols) * cell_w * 2
        block_y = 36 + (pos // cols) * cell_h
        before = Image.open(row['image_path']).convert('L')
        after = Image.open(row['processed_path']).convert('L')
        before.thumbnail((128, 128), Image.Resampling.LANCZOS)
        after.thumbnail((128, 128), Image.Resampling.LANCZOS)
        before_canvas = Image.new('L', (128, 128), 255)
        after_canvas = Image.new('L', (128, 128), 255)
        before_canvas.paste(before, ((128 - before.width) // 2, (128 - before.height) // 2))
        after_canvas.paste(after, ((128 - after.width) // 2, (128 - after.height) // 2))
        sheet.paste(before_canvas.convert('RGB'), (block_x + 8, block_y + 74))
        sheet.paste(after_canvas.convert('RGB'), (block_x + cell_w + 8, block_y + 74))
        label = f"{row.get('query_char', '')} / {row.get('style_label', '')} / {row.get('author', '')}"
        label2 = f"{row.get('image_type', 'unknown')} / {row.get('processing_mode', '')} / wm={row.get('watermark_attempted', False)}"
        draw.text((block_x + 8, block_y + 6), label[:36], fill='black')
        draw.text((block_x + 8, block_y + 24), label2[:46], fill='black')
        draw.text((block_x + 8, block_y + 48), f"dirty={float(row.get('background_dirty_ratio', 0)):.4f}", fill='black')
        draw.text((block_x + 8, block_y + 60), 'before', fill='black')
        draw.text((block_x + cell_w + 8, block_y + 60), 'after', fill='black')
    sheet.save(out_path)
    print(f'Wrote debug sheet: {out_path}')

save_v2_debug_sheet(success_df, OUTPUT_ROOT / 'debug_before_after_100.png', n=100, watermark_only=False)
save_v2_debug_sheet(success_df, OUTPUT_ROOT / 'debug_watermark_cases.png', n=80, watermark_only=True)
